# 환경 설정

In [1]:
# 필요 시만 실행
!pip install -U sentence-transformers transformers keybert umap-learn hdbscan networkx python-dateutil scikit-learn pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 65.1 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dask-cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.3 which is i

In [2]:
import os

# 환경변수에서 API 키 로딩(사용자/설정에서 주입)
TE_API_KEY = os.environ.get('TE_API_KEY')               # Trading Economics API key
FINNHUB_API_KEY = os.environ.get('FINNHUB_API_KEY')     # Finnhub API token
EVENT_REGISTRY_API_KEY = os.environ.get('EVENT_REGISTRY_API_KEY')  # Event Registry API key
NAVER_CLIENT_ID = os.environ.get('NAVER_CLIENT_ID')     # Naver API client ID
NAVER_CLIENT_SECRET = os.environ.get('NAVER_CLIENT_SECRET')  # Naver API client secret
DART_API_KEY = os.environ.get('DART_API_KEY')           # DART (Korean disclosure) API key

# 모든 키가 준비되지 않으면 데모용 더미 모드로 전환
dummy_mode = not (TE_API_KEY and FINNHUB_API_KEY and EVENT_REGISTRY_API_KEY
                  and NAVER_CLIENT_ID and NAVER_CLIENT_SECRET and DART_API_KEY)
print("Dummy mode is", "ON" if dummy_mode else "OFF")


Dummy mode is ON


In [3]:
from __future__ import annotations
import os, re, json, math, time, numpy as np, pandas as pd
from datetime import datetime
from dateutil.parser import isoparse

CFG = dict(
    # 임베딩 모델: E5가 다국어/유사도 안정적. 실패 시 멀티링구얼 mpnet로 폴백.
    embedding_model_primary="intfloat/multilingual-e5-base",
    embedding_model_fallback="sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    use_e5_prefix=True,                 # E5는 "passage: " 접두어 권장

    # 기사 중복 제거 (row 단위)
    dedup_threshold=0.91,               # 코사인 유사도 이상이면 중복으로 제거(E5 모델은 0.91~0.93 추천)

    # 기사 분류(외생/내생) — 제로샷 템플릿 "{}" 필수
    zero_shot_model="joeddav/xlm-roberta-large-xnli",
    class_labels=["외생 변수", "내생 변수"],  # 자연어 라벨
    label_map={"외생 변수":"exo", "내생 변수":"endo"},
    zshot_template="{}에 관한 뉴스다.",
    zshot_conf_thresh=0.60,             # 저신뢰 임계

    # 카테고리별 그래프 클러스터링(보조: 다양성·피처 계산용)
    tau_exo=0.60,                       # 엣지 생성 코사인 임계(exo)
    tau_endo=0.68,                      # 엣지 생성 코사인 임계(endo)

    # 소스 가중치(학습 피처용)
    source_weight={"TradingEconomics":1.0,"Finnhub":0.8,"EventRegistry":0.9,"NaverNews":0.7,"DART":1.0},

    # Weight 회귀
    weight_model="elasticnet",          # elasticnet 한정
    en_alpha=0.01, en_l1=0.2,

    # Polarity 제로샷
    sent_labels=["positive","negative","neutral"],
    sent_template="This news has a {} impact on financial markets.",

    # Span 기본값
    span_max=8,

    # TopN 및 다양성
    weekly_topn_per_cat={"exo":5,"endo":5},
    diversity_lambda=0.5,               # MMR λ
    per_cluster_cap=1,                  # 같은 클러스터에서 뽑을 수 있는 최대 기사 수

    # 출력 정책
    output_append_extras=True,          # 필수 5컬럼 + 보조컬럼 append
    core_contains="all",                # "top_only" | "all"
    out_ex="events_ex.csv",
    out_in="events_in.csv",
)

# 기본 검증
assert 0.5 <= CFG["tau_exo"] <= 0.9 and 0.5 <= CFG["tau_endo"] <= 0.9
assert 0.0 < CFG["dedup_threshold"] < 0.99


# 소스별 뉴스 수집

다음 소스들에서 뉴스를 수집한다.

- Trading Economics – News API: 거시/마켓 뉴스.

- Finnhub – Market News: 전반적 시장 헤드라인(필요 시 개별 기업 뉴스도 가능).

- Event Registry – Top Events/Stories: 전 세계 트렌딩 이벤트/스토리.

- 네이버 뉴스: 한국어 뉴스(국내 거시/증시/기업 보강).

- DART(전자공시): 국내 기업 공시(주요사항보고 등).

각 소스는 날짜 범위(또는 쿼리)에 따라 데이터를 반환하도록 함수를 분리한다. 함수는 실호출(키가 있을 때)과 더미 데이터(키가 없을 때)를 모두 지원한다.

## Trading Economics News API

In [4]:
import requests
from datetime import datetime

def get_tradingeconomics_news(start_date: str, end_date: str):
    """
    Trading Economics API에서 start_date~end_date 사이의 뉴스를 수집.
    'date'와 'title' 필드를 갖는 dict 리스트를 반환.
    """
    if not dummy_mode:
        # API는 YYYY-MM-DD 형식을 사용
        d1 = start_date[:10]
        d2 = end_date[:10]
        url = f"https://api.tradingeconomics.com/news?c={TE_API_KEY}&d1={d1}&d2={d2}"
        resp = requests.get(url)
        if resp.status_code != 200:
            print("TradingEconomics API error:", resp.status_code, resp.text)
            return []
        data = resp.json()
        news_items = []
        for item in data:
            date_str = item.get('date')
            # ISO 8601로 정규화
            if date_str:
                try:
                    dt = datetime.fromisoformat(date_str.replace("Z", "+00:00"))
                    date_iso = dt.strftime("%Y-%m-%dT%H:%M:%SZ")
                except Exception:
                    date_iso = date_str
            else:
                date_iso = start_date + "T00:00:00Z"
            title = item.get('title') or ""
            news_items.append({"date": date_iso, "title": title, "source": "TradingEconomics"})
        return news_items
    else:
        # 데모용 더미 데이터
        dummy_data = [
            {"date": "2025-10-14T00:00:00Z", "title": "Fed signals no more rate hikes", "source": "TradingEconomics"},
            {"date": "2025-10-15T00:00:00Z", "title": "Oil prices surge on supply concerns", "source": "TradingEconomics"},
            {"date": "2025-10-15T00:00:00Z", "title": "China's GDP growth slows to 4%", "source": "TradingEconomics"},
            {"date": "2025-10-12T00:00:00Z", "title": "Jenson Hwang, Nvidia's Chef, Needs more HBM3 Semiconductor.", "source": "TradingEconomics"},
            {"date": "2025-10-13T00:00:00Z", "title": "Tesla crashed 2024`s Car sales record", "source": "TradingEconomics"}
        ]
        return dummy_data

# 사용 예시:
te_news = get_tradingeconomics_news("2025-10-13", "2025-10-17")
print(f"TradingEconomics returned {len(te_news)} items. Sample 1st item:\n", te_news[0] if te_news else "(no data)")


TradingEconomics returned 5 items. Sample 1st item:
 {'date': '2025-10-14T00:00:00Z', 'title': 'Fed signals no more rate hikes', 'source': 'TradingEconomics'}


## Finnhub Market News API

In [5]:
def get_finnhub_news():
    """
    Finnhub에서 일반 시장 뉴스를 최신순으로 조회.
    'date'와 'title' 필드의 리스트 반환.
    """
    if not dummy_mode:
        url = f"https://finnhub.io/api/v1/news?category=general&token={FINNHUB_API_KEY}"
        resp = requests.get(url)
        if resp.status_code != 200:
            print("Finnhub API error:", resp.status_code, resp.text)
            return []
        data = resp.json()
        news_items = []
        for item in data:
            # Finnhub는 'datetime'에 유닉스 타임스탬프를 제공
            ts = item.get('datetime')
            if ts:
                dt = datetime.utcfromtimestamp(ts)
                date_iso = dt.strftime("%Y-%m-%dT%H:%M:%SZ")
            else:
                date_iso = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
            title = item.get('headline') or item.get('title') or ""
            news_items.append({"date": date_iso, "title": title, "source": "Finnhub"})
        return news_items
    else:
        # 데모용 더미 데이터
        dummy_data = [
            {"date": "2025-10-13T00:00:00Z", "title": "Middle East conflict rattles markets", "source": "Finnhub"},
            {"date": "2025-10-14T00:00:00Z", "title": "Fed likely to pause interest rate hikes", "source": "Finnhub"},
            {"date": "2025-10-15T00:00:00Z", "title": "Oil jumps as supply worries mount", "source": "Finnhub"}
        ]
        return dummy_data

# 사용 예시:
fh_news = get_finnhub_news()
print(f"Finnhub returned {len(fh_news)} items. Sample:\n", fh_news[0] if fh_news else "(no data)")


Finnhub returned 3 items. Sample:
 {'date': '2025-10-13T00:00:00Z', 'title': 'Middle East conflict rattles markets', 'source': 'Finnhub'}


## Event Registry Top Events

In [6]:
def get_eventregistry_top_events():
    """
    Event Registry API에서 트렌딩 이벤트를 조회.
    'date'와 'title' 필드의 리스트 반환.
    """
    if not dummy_mode:
        # 설명용 가상 엔드포인트 예시(실 서비스에서는 문서의 실제 엔드포인트를 사용)
        url = f"http://eventregistry.org/api/v1/event/getTrendingEvents?apiKey={EVENT_REGISTRY_API_KEY}"
        resp = requests.get(url)
        if resp.status_code != 200:
            print("Event Registry API error:", resp.status_code, resp.text)
            return []
        data = resp.json()
        events = []
        for ev in data.get('events', []):
            date_iso = ev.get('eventDate') or ev.get('date') or datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
            title = ev.get('title', {}).get('eng') or ev.get('title') or ""
            events.append({"date": date_iso, "title": title, "source": "EventRegistry"})
        return events
    else:
        # 데모용 더미 데이터
        dummy_data = [
            {"date": "2025-10-13T00:00:00Z", "title": "Escalation in Middle East conflict", "source": "EventRegistry"},
            {"date": "2025-10-14T00:00:00Z", "title": "Fed indicates rate hike cycle ending", "source": "EventRegistry"}
        ]
        return dummy_data

# 사용 예시:
er_events = get_eventregistry_top_events()
print(f"EventRegistry returned {len(er_events)} items. Sample:\n", er_events[0] if er_events else "(no data)")


EventRegistry returned 2 items. Sample:
 {'date': '2025-10-13T00:00:00Z', 'title': 'Escalation in Middle East conflict', 'source': 'EventRegistry'}


## 네이버 뉴스 API

In [7]:
import re

def get_naver_news(query: str, max_results: int = 10):
    """
    네이버 뉴스 검색 API로 query에 해당하는 최신 뉴스 조회.
    """
    if not dummy_mode:
        url = f"https://openapi.naver.com/v1/search/news.json?query={query}&display={max_results}&sort=date"
        headers = {
            "X-Naver-Client-Id": NAVER_CLIENT_ID,
            "X-Naver-Client-Secret": NAVER_CLIENT_SECRET
        }
        resp = requests.get(url, headers=headers)
        if resp.status_code != 200:
            print("Naver API error:", resp.status_code, resp.text)
            return []
        data = resp.json()
        items = data.get('items', [])
        news_items = []
        for it in items:
            # 제목의 HTML 태그 제거
            title_html = it.get('title', '')
            title = re.sub('<[^<]+?>', '', title_html)
            # pubDate를 ISO로 변환(예: "Tue, 20 Oct 2025 12:00:00 +0900")
            pub_date = it.get('pubDate')
            if pub_date:
                try:
                    dt = datetime.strptime(pub_date, "%a, %d %b %Y %H:%M:%S %z")
                    date_iso = dt.astimezone(datetime.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
                except Exception:
                    date_iso = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
            else:
                date_iso = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
            news_items.append({"date": date_iso, "title": title, "source": "NaverNews"})
        return news_items
    else:
        # 데모용 더미 데이터
        dummy_data_map = {
            "경제": [
                {"date": "2025-10-17T00:00:00Z", "title": "한은 기준금리 동결", "source": "NaverNews"}
            ],
            "증시": [
                {"date": "2025-10-13T00:00:00Z", "title": "삼성전자 3분기 실적 호조", "source": "NaverNews"},
                {"date": "2025-10-14T00:00:00Z", "title": "현대차 신규 전기차 모델 출시 발표", "source": "NaverNews"},
                {"date": "2025-10-15T00:00:00Z", "title": "롯데그룹 회장 검찰 수사", "source": "NaverNews"},
                {"date": "2025-10-16T00:00:00Z", "title": "카카오 1조원 유상증자 결정", "source": "NaverNews"},
                {"date": "2025-10-17T00:00:00Z", "title": "LG에너지솔루션 1조원 배당 발표", "source": "NaverNews"}
            ]
        }
        return dummy_data_map.get(query, [])[:max_results]

# 사용 예시:
naver_news_econ = get_naver_news("경제", max_results=5)   # 거시/경제
naver_news_stocks = get_naver_news("증시", max_results=5) # 증시/기업
naver_news = naver_news_econ + naver_news_stocks
print(f"Naver News returned {len(naver_news)} items. Samples:\n", naver_news[:2])


Naver News returned 6 items. Samples:
 [{'date': '2025-10-17T00:00:00Z', 'title': '한은 기준금리 동결', 'source': 'NaverNews'}, {'date': '2025-10-13T00:00:00Z', 'title': '삼성전자 3분기 실적 호조', 'source': 'NaverNews'}]


## DART(전자공시)

In [8]:
def get_dart_disclosures(start_date: str, end_date: str):
    """
    DART Open API에서 start_date~end_date 사이의 공시 조회.
    '주요사항보고서' 중심으로 필터하여 이벤트 리스트 반환.
    """
    if not dummy_mode:
        params = {
            "crtfc_key": DART_API_KEY,
            "bgn_de": start_date.replace("-", ""),
            "end_de": end_date.replace("-", ""),
            "page_no": 1,
            "page_count": 100
        }
        resp = requests.get("https://opendart.fss.or.kr/api/list.json", params=params)
        if resp.status_code != 200:
            print("DART API error:", resp.status_code, resp.text)
            return []
        data = resp.json()
        if data.get("status") != "000":
            print("DART API returned status", data.get("status"), data.get("message"))
            return []
        items = data.get("list", [])
        events = []
        for it in items:
            report_nm = it.get("report_nm", "")
            # 주요사항보고서만 선별
            if "주요사항보고서" in report_nm:
                rcept_dt = it.get("rcept_dt")  # 예: "20251016"
                if rcept_dt:
                    try:
                        dt = datetime.strptime(rcept_dt, "%Y%m%d")
                        date_iso = dt.strftime("%Y-%m-%dT00:00:00Z")
                    except Exception:
                        date_iso = start_date + "T00:00:00Z"
                else:
                    date_iso = start_date + "T00:00:00Z"
                corp = it.get("corp_name") or ""
                if corp:
                    if "보고서" in report_nm:
                        issue = report_nm.split("보고서")[-1].strip("()")
                        title = f"{corp} {issue}"
                    else:
                        title = f"{corp} {report_nm}"
                else:
                    title = report_nm
                events.append({"date": date_iso, "title": title, "source": "DART"})
        return events
    else:
        # 데모용 더미 데이터
        dummy_data = [
            {"date": "2025-10-16T00:00:00Z", "title": "주요사항보고서(유상증자결정)", "corp": "카카오", "source": "DART"},
            {"date": "2025-10-17T00:00:00Z", "title": "주요사항보고서(현금배당결정)", "corp": "LG에너지솔루션", "source": "DART"}
        ]
        for item in dummy_data:
            if item.get("corp"):
                issue = item["title"].split("보고서")[-1].strip("()")
                item["title"] = f"{item['corp']} {issue}"
                item.pop("corp", None)
        return dummy_data

# 사용 예시:
dart_events = get_dart_disclosures("2025-10-13", "2025-10-17")
print(f"DART returned {len(dart_events)} items. Sample:\n", dart_events[0] if dart_events else "(no data)")


DART returned 2 items. Sample:
 {'date': '2025-10-16T00:00:00Z', 'title': '카카오 유상증자결정', 'source': 'DART'}


## 전체 소스 수집 통합

In [9]:
# 관심 기간(예시: 1주)
start_date = "2025-10-13"
end_date   = "2025-10-17"

# 소스별 수집
te_news = get_tradingeconomics_news(start_date, end_date)
fh_news = get_finnhub_news()
er_events = get_eventregistry_top_events()
# 네이버는 거시/증시 두 쿼리로 보강
naver_news = get_naver_news("경제", max_results=5) + get_naver_news("증시", max_results=5)
dart_events = get_dart_disclosures(start_date, end_date)

# 통합
combined_news = te_news + fh_news + er_events + naver_news + dart_events
print("Total news items collected:", len(combined_news))

Total news items collected: 18


In [10]:
# 검증
print(type(combined_news))
print(len(combined_news))
combined_news

<class 'list'>
18


[{'date': '2025-10-14T00:00:00Z',
  'title': 'Fed signals no more rate hikes',
  'source': 'TradingEconomics'},
 {'date': '2025-10-15T00:00:00Z',
  'title': 'Oil prices surge on supply concerns',
  'source': 'TradingEconomics'},
 {'date': '2025-10-15T00:00:00Z',
  'title': "China's GDP growth slows to 4%",
  'source': 'TradingEconomics'},
 {'date': '2025-10-12T00:00:00Z',
  'title': "Jenson Hwang, Nvidia's Chef, Needs more HBM3 Semiconductor.",
  'source': 'TradingEconomics'},
 {'date': '2025-10-13T00:00:00Z',
  'title': 'Tesla crashed 2024`s Car sales record',
  'source': 'TradingEconomics'},
 {'date': '2025-10-13T00:00:00Z',
  'title': 'Middle East conflict rattles markets',
  'source': 'Finnhub'},
 {'date': '2025-10-14T00:00:00Z',
  'title': 'Fed likely to pause interest rate hikes',
  'source': 'Finnhub'},
 {'date': '2025-10-15T00:00:00Z',
  'title': 'Oil jumps as supply worries mount',
  'source': 'Finnhub'},
 {'date': '2025-10-13T00:00:00Z',
  'title': 'Escalation in Middle East 

# (효율화를 위한) 영문화 번역 및 정규화 + 임베딩

In [11]:
from sentence_transformers import SentenceTransformer

def to_english(text: str) -> str:
    # 이미 번역 파이프라인이 있다면 여기서 호출하라. 없으면 원문 사용.
    return text

def _load_model():
    try:
        m = SentenceTransformer(CFG["embedding_model_primary"])
        return m, True
    except Exception:
        m = SentenceTransformer(CFG["embedding_model_fallback"])
        return m, False

embed_model, using_e5 = _load_model()
def embed_texts(texts):
    if using_e5 and CFG["use_e5_prefix"]:
        texts = ["passage: "+t for t in texts]
    return embed_model.encode(texts, normalize_embeddings=True, show_progress_bar=False)

# 영문화된 타이틀
titles_all = [to_english(x["title"]) for x in combined_news]
E_all = embed_texts(titles_all)
print("Embedding shape:", E_all.shape, "E5:", using_e5)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Embedding shape: (18, 768) E5: True


- 검증 체크포인트: (N,d) 일치, 평균 노름 ≈ 1.
- 튜닝 포인트: E5 사용 시 "passage: " prefix on/off 비교.

In [12]:
titles_all

['Fed signals no more rate hikes',
 'Oil prices surge on supply concerns',
 "China's GDP growth slows to 4%",
 "Jenson Hwang, Nvidia's Chef, Needs more HBM3 Semiconductor.",
 'Tesla crashed 2024`s Car sales record',
 'Middle East conflict rattles markets',
 'Fed likely to pause interest rate hikes',
 'Oil jumps as supply worries mount',
 'Escalation in Middle East conflict',
 'Fed indicates rate hike cycle ending',
 '한은 기준금리 동결',
 '삼성전자 3분기 실적 호조',
 '현대차 신규 전기차 모델 출시 발표',
 '롯데그룹 회장 검찰 수사',
 '카카오 1조원 유상증자 결정',
 'LG에너지솔루션 1조원 배당 발표',
 '카카오 유상증자결정',
 'LG에너지솔루션 현금배당결정']

In [13]:
# 3.V — Embedding sanity
import numpy as np
assert E_all.shape[0] == len(combined_news), "임베딩 N≠기사수"
# 정규화 확인(평균 norm ~1)
norms = np.linalg.norm(E_all, axis=1)
print(f"[3.V] shape={E_all.shape}, norm(mean)={norms.mean():.3f}, E5={using_e5}")
assert 0.95 <= norms.mean() <= 1.05, "정규화 임베딩이 아님(모델/옵션 확인)"


[3.V] shape=(18, 768), norm(mean)=1.000, E5=True


# 유사 기사 필터링/클러스터링(임베딩 + 중복 제거)
 단위 : 개별 기사

In [14]:
from sentence_transformers import util

def dedup_by_threshold(items, E, threshold: float):
    kept = []
    used = np.zeros(len(items), dtype=bool)
    for i in range(len(items)):
        if used[i]:
            continue
        kept.append(items[i])
        sims = util.cos_sim(E[i], E).cpu().numpy().ravel()
        dup_idx = np.where((sims >= threshold) & (~used))[0]
        used[dup_idx] = True
    return kept, used

dedup_news, used_mask = dedup_by_threshold(combined_news, E_all, CFG["dedup_threshold"])
print(f"Dedup: {len(combined_news)} -> {len(dedup_news)} kept")

# 검증: 잔존 pairwise 유사도 체크(샘플)
if len(dedup_news) > 1:
    E_dedup = embed_texts([to_english(x["title"]) for x in dedup_news])
    S = util.cos_sim(E_dedup, E_dedup).cpu().numpy()
    np.fill_diagonal(S, -1)
    print("max cosine among deduped:", float(S.max()))


Dedup: 18 -> 13 kept
max cosine among deduped: 0.9057720303535461


- 검증 체크포인트: 잔존 최대 유사도 < 임계. 드롭율 0–60% 사이가 보통.
- 튜닝 포인트: 중복 과다→ threshold↑, 과소→ threshold↓, 제목 정규화 강화.

In [15]:
dedup_news

[{'date': '2025-10-14T00:00:00Z',
  'title': 'Fed signals no more rate hikes',
  'source': 'TradingEconomics'},
 {'date': '2025-10-15T00:00:00Z',
  'title': 'Oil prices surge on supply concerns',
  'source': 'TradingEconomics'},
 {'date': '2025-10-15T00:00:00Z',
  'title': "China's GDP growth slows to 4%",
  'source': 'TradingEconomics'},
 {'date': '2025-10-12T00:00:00Z',
  'title': "Jenson Hwang, Nvidia's Chef, Needs more HBM3 Semiconductor.",
  'source': 'TradingEconomics'},
 {'date': '2025-10-13T00:00:00Z',
  'title': 'Tesla crashed 2024`s Car sales record',
  'source': 'TradingEconomics'},
 {'date': '2025-10-13T00:00:00Z',
  'title': 'Middle East conflict rattles markets',
  'source': 'Finnhub'},
 {'date': '2025-10-14T00:00:00Z',
  'title': 'Fed likely to pause interest rate hikes',
  'source': 'Finnhub'},
 {'date': '2025-10-13T00:00:00Z',
  'title': 'Escalation in Middle East conflict',
  'source': 'EventRegistry'},
 {'date': '2025-10-17T00:00:00Z',
  'title': '한은 기준금리 동결',
  'sou

In [16]:
# 4.V — Dedup sanity
from sentence_transformers import util
N0, N1 = len(combined_news), len(dedup_news)
print(f"[4.V] dedup {N0} -> {N1} (drop={(N0-N1)/max(1,N0):.1%})")
if N1 > 1:
    E_d = embed_texts([to_english(x["title"]) for x in dedup_news])
    S = util.cos_sim(E_d, E_d).cpu().numpy()
    np.fill_diagonal(S, -1)
    mx = float(S.max())
    print("max cosine among kept:", round(mx,3))
    assert mx < CFG["dedup_threshold"] + 1e-3, "중복 제거 실패"


[4.V] dedup 18 -> 13 (drop=27.8%)
max cosine among kept: 0.906


# 제로샷 분류(외생/내생) — 기사 단위

In [17]:
# === CELL 5 (REPLACE) — Content-only weak supervision teachers + student classifier ===
import numpy as np, pandas as pd, re
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sentence_transformers import util
from transformers import pipeline

titles_en = [to_english(x["title"]) for x in dedup_news]
E = embed_texts(titles_en)  # (N, d)

# ---------- T1: 제로샷 NLI (영문 라벨) ----------
zshot = pipeline("zero-shot-classification", model=CFG["zero_shot_model"])
labels_en = [
    "exogenous macro/policy/geopolitics/commodities factors (not company-specific)",
    "company-specific events (earnings, funding, products, investigations, deals)"
]
_label_map = {labels_en[0]:"exo", labels_en[1]:"endo"}

def zs_endo_prob(text: str) -> float:
    r = zshot(sequences=text, candidate_labels=labels_en,
              hypothesis_template="This headline is about {}.")
    return float([s for l,s in zip(r["labels"], r["scores"]) if _label_map[l]=="endo"][0])

p_zs = np.array([zs_endo_prob(t) for t in titles_en])  # [0,1]

# ---------- T2: NER 기반 구조 신호(사전 X) ----------
# spaCy 멀티랭 NER 사용(설치 필요: en_core_web_sm, ko_core_news_sm가 있으면 우선 사용)
try:
    import spacy
    try:
        nlp_en = spacy.load("en_core_web_sm")
    except Exception:
        nlp_en = spacy.load("xx_ent_wiki_sm")
    try:
        nlp_ko = spacy.load("ko_core_news_sm")
    except Exception:
        nlp_ko = spacy.load("xx_ent_wiki_sm")
except Exception:
    nlp_en = nlp_ko = None

def lang_of(s:str):
    # 아주 단순: 한글 포함 여부
    return "ko" if re.search(r"[가-힣]", s) else "en"

def ner_counts(text:str):
    if nlp_en is None or nlp_ko is None:
        return dict(ORG=0, GPE=0, PERSON=0, NORP=0, EVENT=0, MONEY=0)
    doc = (nlp_ko(text) if lang_of(text)=="ko" else nlp_en(text))
    c = {"ORG":0,"GPE":0,"PERSON":0,"NORP":0,"EVENT":0,"MONEY":0}
    for e in doc.ents:
        if e.label_ in c: c[e.label_] += 1
    return c

def structural_score(text:str):
    # 티커/심볼 패턴(사전 X, 순수 패턴)
    has_ticker = 1.0 if re.search(r"\b[A-Z]{1,5}\b", text) else 0.0
    has_pct    = 1.0 if re.search(r"\b\d+(\.\d+)?%\b", text) else 0.0
    has_money  = 1.0 if re.search(r"[$€₩¥]\s?\d", text) else 0.0
    return has_ticker, has_pct, has_money

# 엔티티/구조를 조합해 "endo 확률 비슷한 점수" 산출 (사전이 아니라 통계적 구조 신호)
def ner_endo_prob(text:str):
    c = ner_counts(text)
    tk, pct, money = structural_score(text)
    # 회사지향 신호: ORG + (티커/금액/퍼센트), 거시지향 신호: GPE/NORP/EVENT
    comp = c["ORG"] + 0.5*tk + 0.5*money + 0.3*pct
    macro = c["GPE"] + 0.5*c["NORP"] + 0.5*c["EVENT"]
    # 시그모이드 유사 변환
    val = comp - macro
    return float(1/(1+np.exp(-val)))  # 0~1

p_ner = np.array([ner_endo_prob(t) for t in titles_en])

# ---------- T3: 임베딩 군집 프로토타입 ----------
def graph_cluster_idx(E, tau=0.62):
    n = E.shape[0]
    if n<=1: return {0:[0]} if n==1 else {}
    S = util.cos_sim(E, E).cpu().numpy()
    g = {i:set() for i in range(n)}
    for i in range(n):
        for j in range(i+1,n):
            if S[i,j] >= tau: g[i].add(j); g[j].add(i)
    # 연결요소
    seen=set(); comps=[]
    for i in range(n):
        if i in seen: continue
        stack=[i]; comp=[]
        while stack:
            v=stack.pop()
            if v in seen: continue
            seen.add(v); comp.append(v)
            stack.extend(list(g[v]-seen))
        comps.append(sorted(comp))
    return {k:v for k,v in enumerate(comps)}

def cluster_proto_endo_prob(E, titles, tau=0.62):
    comps = graph_cluster_idx(E, tau=tau)
    # 각 클러스터에서 NER 기반 endo 비율이 높으면 endo 클러스터로 간주
    p = np.zeros(len(titles), dtype=float)
    for _, idxs in comps.items():
        if not idxs: continue
        pr = np.mean([ner_endo_prob(titles[i]) for i in idxs])
        for i in idxs:
            p[i] = pr
    return p

p_proto = cluster_proto_endo_prob(E, titles_en, tau=0.62)

# ---------- 라벨 합성(콘텐츠 전용) ----------
# 고신뢰 채택 기준: 세 교사 평균/합의가 강한 샘플만 학생 훈련에 사용
w_zs, w_ner, w_proto = 0.45, 0.15, 0.45  # 합=1
p_mix = w_zs*p_zs + w_ner*p_ner + w_proto*p_proto

train_X, train_y = [], []
for i, p in enumerate(p_mix):
    # 강한 확신만 채택 (endo>=0.75 또는 <=0.25)
    if p >= 0.75:
        train_X.append(E[i]); train_y.append(1)
    elif p <= 0.25:
        train_X.append(E[i]); train_y.append(0)

train_X = np.array(train_X)
train_y = np.array(train_y) if len(train_X) else np.array([])

print(f"[Teachers] pseudo-labeled train set: {len(train_y)} / {len(dedup_news)}")

# ---------- 학생(최종 추론기): 임베딩 → 로지스틱 ----------
if len(train_y) >= 6 and len(np.unique(train_y))==2:
    student = Pipeline([
        ("sc", StandardScaler(with_mean=False)),
        ("lr", LogisticRegression(max_iter=1000, class_weight="balanced", C=2.0))
    ])
    student.fit(train_X, train_y)
    p_student = student.predict_proba(E)[:,1]  # endo 확률
else:
    # 데이터 적으면 teacher 혼합 그대로 사용
    student = None
    p_student = p_mix
    print("[Teachers] not enough confident samples; using teacher mixture directly.")

# ---------- 최종 결정 ----------
def fuse(p_s, p_mix, p_zs, hi=0.58, lo=0.42, blend=0.4):
    # 학생이 확신적이면 사용, 아니면 teacher 혼합과 제로샷을 블렌딩
    base = p_s if (p_s>=hi or p_s<=lo) else (blend*p_s + (1-blend)*p_mix)
    # 마지막으로 제로샷과 살짝 평균(과신 완화)
    return 0.7*base + 0.3*p_zs

p_final = np.array([fuse(p_student[i], p_mix[i], p_zs[i]) for i in range(len(dedup_news))])
cats   = np.where(p_final>=0.5, "endo", "exo")
scores = np.where(p_final>=0.5, p_final, 1-p_final)

rows = [{**dedup_news[i], "cat":cats[i], "cat_score":float(scores[i])} for i in range(len(dedup_news))]



config.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Device set to use cpu


[Teachers] pseudo-labeled train set: 0 / 13
[Teachers] not enough confident samples; using teacher mixture directly.


In [18]:
rows

[{'date': '2025-10-14T00:00:00Z',
  'title': 'Fed signals no more rate hikes',
  'source': 'TradingEconomics',
  'cat': np.str_('exo'),
  'cat_score': 0.6435404685197035},
 {'date': '2025-10-15T00:00:00Z',
  'title': 'Oil prices surge on supply concerns',
  'source': 'TradingEconomics',
  'cat': np.str_('exo'),
  'cat_score': 0.7358294979912797},
 {'date': '2025-10-15T00:00:00Z',
  'title': "China's GDP growth slows to 4%",
  'source': 'TradingEconomics',
  'cat': np.str_('exo'),
  'cat_score': 0.5404492877729783},
 {'date': '2025-10-12T00:00:00Z',
  'title': "Jenson Hwang, Nvidia's Chef, Needs more HBM3 Semiconductor.",
  'source': 'TradingEconomics',
  'cat': np.str_('endo'),
  'cat_score': 0.6875154749246677},
 {'date': '2025-10-13T00:00:00Z',
  'title': 'Tesla crashed 2024`s Car sales record',
  'source': 'TradingEconomics',
  'cat': np.str_('endo'),
  'cat_score': 0.6317004621716579},
 {'date': '2025-10-13T00:00:00Z',
  'title': 'Middle East conflict rattles markets',
  'source': 

- 검증 체크포인트: 평균 score ≥0.6 권장, DART 불일치 최소화.
- 튜닝 포인트:
  - alpha(사전 가중): 0.4–0.8 사이 탐색. 값↑ → 규칙(사전) 영향↑.

  - conf_margin: 0.10–0.20. 박빙일 때 DART/COMP 우선으로 전환되는 범위.

  - labels_kor 문구를 더 구체화하면 Exo/Endo 분리가 더 잘 된다.

  - (필요 시) 분류 입력을 to_english()로 영문화하고 동일 라벨의 영문 버전도 테스트.

In [19]:
# 5.V — Zero-shot (hybrid) sanity
import pandas as pd, numpy as np
cats = pd.Series([r["cat"] for r in rows]).value_counts().to_dict()
scores = [r["cat_score"] for r in rows]
print(f"[5.V] cat_dist={cats}, avg_score={np.mean(scores):.3f}")
dart_bad = [r for r in rows if r["source"]=="DART" and r["cat"]!="endo"]
print(f"DART mismatches: {len(dart_bad)}")
assert all(r["cat"] in {"exo","endo"} for r in rows)


[5.V] cat_dist={np.str_('exo'): 7, np.str_('endo'): 6}, avg_score=0.640
DART mismatches: 0


# 카테고리별 그래프 클러스터(보조)

In [20]:
import networkx as nx

def graph_cluster(items, tau: float):
    if len(items) <= 1:
        return {0: items} if items else {}
    titles = [to_english(x["title"]) for x in items]
    E = embed_texts(titles)
    S = util.cos_sim(E, E).cpu().numpy()
    np.fill_diagonal(S, 0.0)
    G = nx.Graph()
    G.add_nodes_from(range(len(items)))
    for i in range(len(items)):
        for j in range(i+1, len(items)):
            if S[i,j] >= tau:
                G.add_edge(i,j, w=float(S[i,j]))
    comps = list(nx.connected_components(G))
    return {k: [items[i] for i in sorted(comp)] for k, comp in enumerate(comps)}

rows_exo  = [r for r in rows if r["cat"]=="exo"]
rows_endo = [r for r in rows if r["cat"]=="endo"]

clusters_exo  = graph_cluster(rows_exo,  CFG["tau_exo"])
clusters_endo = graph_cluster(rows_endo, CFG["tau_endo"])

def cohesion(items):
    if len(items)<2: return math.nan
    E = embed_texts([to_english(x["title"]) for x in items])
    S = util.cos_sim(E,E).cpu().numpy()
    np.fill_diagonal(S, math.nan)
    return float(np.nanmean(S))

def report_clusters(cldict, name):
    sizes = [len(v) for v in cldict.values()]
    cohs  = [cohesion(v) for v in cldict.values() if len(v)>=2]
    print(f"[{name}] k={len(cldict)}, sizes={sizes}, cohesion_mean={np.nanmean(cohs) if cohs else 'NA'}")

report_clusters(clusters_exo,  "EXO")
report_clusters(clusters_endo, "ENDO")


[EXO] k=1, sizes=[7], cohesion_mean=0.7994195818901062
[ENDO] k=1, sizes=[6], cohesion_mean=0.8277040123939514


- 검증 체크포인트: 각 클러스터 평균 코사인 ≥ tau 근처/이상이면 정상.
- 튜닝 포인트: 과분할→ tau↓(0.02–0.05), 과대묶음→ tau↑.

In [21]:
# 6.V — Graph clustering cohesion
def cohesion(items):
    if len(items)<2: return np.nan
    E = embed_texts([to_english(x["title"]) for x in items])
    S = util.cos_sim(E,E).cpu().numpy()
    np.fill_diagonal(S, np.nan)
    return float(np.nanmean(S))

def report(name, cldict):
    sizes = [len(v) for v in cldict.values()]
    cohs  = [cohesion(v) for v in cldict.values() if len(v)>=2]
    print(f"[6.V] {name}: k={len(cldict)}, sizes={sizes}, cohesion_mean={np.nanmean(cohs) if cohs else 'NA'}")

report("EXO",  clusters_exo)
report("ENDO", clusters_endo)


[6.V] EXO: k=1, sizes=[7], cohesion_mean=0.7994195818901062
[6.V] ENDO: k=1, sizes=[6], cohesion_mean=0.8277040123939514


# 스코어링(데이터 기반)

### 클러스터 피처(centroid/size/min_date) + 기사 피처(centrality 등)

In [29]:
from sentence_transformers import util

def cluster_features(cldict):
    feats = {}
    for cid, items in cldict.items():
        titles = [to_english(x["title"]) for x in items]
        E = embed_texts(titles)
        cen = E.mean(axis=0)
        size = len(items)
        min_date = min(x["date"] for x in items)
        feats[cid] = dict(centroid=cen, size=size, min_date=min_date)
    return feats

cf_exo  = cluster_features(clusters_exo)
cf_endo = cluster_features(clusters_endo)

def src_w(src): return CFG["source_weight"].get(src, 0.5)

row_feats = []
def add_rows(cldict, cfeat, cat):
    for cid, items in cldict.items():
        cen = cfeat[cid]["centroid"]
        for it in items:
            E = embed_texts([to_english(it["title"])])[0]
            centrality = float(util.cos_sim(E, cen))
            row_feats.append(dict(
                cluster_id=f"{cat}-{cid}",
                date=it["date"], title=it["title"], source=it["source"],
                cat=cat, cat_score=it["cat_score"],
                clust_size=cfeat[cid]["size"],
                src_w=src_w(it["source"]),
                centrality=centrality,
                title_len=len(it["title"]),
            ))

add_rows(clusters_exo,  cf_exo,  "exo")
add_rows(clusters_endo, cf_endo, "endo")

pd.DataFrame(row_feats).head()


,cluster_id,date,title,source,cat,cat_score,clust_size,src_w,centrality,title_len
0,exo-0,2025-10-14T00:00:00Z,Fed signals no more rate hikes,TradingEconomics,exo,0.643540,7,1.0,0.912564,30
1,exo-0,2025-10-15T00:00:00Z,Oil prices surge on supply concerns,TradingEconomics,exo,0.735829,7,1.0,0.907312,35
2,exo-0,2025-10-15T00:00:00Z,China's GDP growth slows to 4%,TradingEconomics,exo,0.540449,7,1.0,0.894482,30
3,exo-0,2025-10-13T00:00:00Z,Middle East conflict rattles markets,Finnhub,exo,0.754774,7,0.8,0.914919,36
4,exo-0,2025-10-14T00:00:00Z,Fed likely to pause interest rate hikes,Finnhub,exo,0.630609,7,0.8,0.933682,39
5,exo-0,2025-10-13T00:00:00Z,Escalation in Middle East conflict,EventRegistry,exo,0.713426,7,0.9,0.924728,34
6,exo-0,2025-10-17T00:00:00Z,한은 기준금리 동결,NaverNews,exo,0.518232,7,0.7,0.882212,10
7,endo-0,2025-10-12T00:00:00Z,"Jenson Hwang, Nvidia's Chef, Needs more HBM3 S...",TradingEconomics,endo,0.687515,6,1.0,0.872583,59
8,endo-0,2025-10-13T00:00:00Z,Tesla crashed 2024`s Car sales record,TradingEconomics,endo,0.631700,6,1.0,0.890658,37
9,endo-0,2025-10-13T00:00:00Z,삼성전자 3분기 실적 호조,NaverNews,endo,0.681374,6,0.7,0.955278,14


- 검증 체크포인트: centrality ∈ [-1,1], src_w 기대 범위(0.5–1.0).
- 튜닝 포인트: source_weight 재조정(내부 지표 반영 시).

In [23]:
# 7.V — Row feature sanity
df_rf = pd.DataFrame(row_feats)
print(df_rf[["cat","clust_size","src_w","centrality","title_len"]].describe())
assert (df_rf["centrality"].between(-1,1)).all(), "centrality 범위 오류"
# 중심성 높은 기사가 같은 클러스터 내 평균보다 높은지 샘플 체크
sample = df_rf.groupby("cluster_id")["centrality"].mean().mean()
print(f"[7.V] avg centrality overall={df_rf['centrality'].mean():.3f}, by-cluster-mean(avg)={sample:.3f}")


       clust_size      src_w  centrality  title_len
count   13.000000  13.000000   13.000000  13.000000
mean     6.538462   0.846154    0.917114  28.538462
std      0.518875   0.139137    0.026891  13.878133
min      6.000000   0.700000    0.872583  10.000000
25%      6.000000   0.700000    0.894482  15.000000
50%      7.000000   0.800000    0.914919  30.000000
75%      7.000000   1.000000    0.938164  36.000000
max      7.000000   1.000000    0.955278  59.000000
[7.V] avg centrality overall=0.917, by-cluster-mean(avg)=0.918


## Weight 학습 모델(샘플: ElasticNet)

초기에는 규칙 없이 표준화된 피처 → 회귀로 Weight 추정.
라벨이 없으면 약지도: “클러스터 크기/출처가중/커버리지” 가중합으로 초기 의사레이블 생성 후 점진 학습.

In [24]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
import statistics as st

if len(row_feats) >= 3:
    Smax = max(x["clust_size"] for x in row_feats)
    Cmax = max(1e-6, max(x["centrality"] for x in row_feats))
    def pseudo_w(f):
        return 0.5*(f["clust_size"]/Smax) + 0.3*(f["src_w"]/1.0) + 0.2*(f["centrality"]/Cmax)

    X = [[f["clust_size"], f["src_w"], f["centrality"], f["title_len"]] for f in row_feats]
    y = [pseudo_w(f) for f in row_feats]
    sc = StandardScaler().fit(X)
    Xn = sc.transform(X)
    reg = ElasticNet(alpha=CFG["en_alpha"], l1_ratio=CFG["en_l1"], random_state=42).fit(Xn, y)
    y_hat = reg.predict(Xn)
    for f, y_ in zip(row_feats, y_hat):
        f["weight"] = float(np.clip(y_, 0.0, 1.0))
    w = np.array([f["weight"] for f in row_feats])
    R = np.corrcoef(np.array(y), w)[0,1] if len(w)>1 else math.nan
    print(f"weight min/mean/max = {w.min():.3f}/{w.mean():.3f}/{w.max():.3f}, corr(pseudo,fit)={R:.3f}")
else:
    # 샘플 부족시 보수적 규칙
    for f in row_feats:
        f["weight"] = float(np.clip(0.5*(f["clust_size"]/max(1,max(x['clust_size'] for x in row_feats))) + 0.5*(f["src_w"]/1.0), 0, 1))


weight min/mean/max = 0.839/0.913/0.984, corr(pseudo,fit)=0.998


- 검증 체크포인트: weight 0–1, 평균 0.3–0.7 권장.
- 튜닝 포인트: en_alpha/l1_ratio 스윕, 피처 추가(novelty, source_coverage, recency).

In [25]:
# 8.V — Weight fit sanity
import numpy as np
w = np.array([f["weight"] for f in row_feats])
assert (w>=0).all() and (w<=1).all(), "weight 범위 오류"
print(f"[8.V] weight min/mean/max = {w.min():.3f}/{w.mean():.3f}/{w.max():.3f}")
# (선택) 의사레이블 상관 R
try:
    R = np.corrcoef(np.array([0]*len(w)) + 1, w)[0,1]  # placeholder to keep cell safe if pseudo not in scope
except Exception:
    R = np.nan
print(f"[8.V] corr(pseudo,weight)≈(상위 셀 출력 참조)")


[8.V] weight min/mean/max = 0.839/0.913/0.984
[8.V] corr(pseudo,weight)≈(상위 셀 출력 참조)


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


## Polarity(방향) — 제로샷/금융감성 + LLM 보조

In [26]:
def polarity_score(title: str) -> float:
    r = zshot(sequences=title, candidate_labels=CFG["sent_labels"], hypothesis_template=CFG["sent_template"])
    lab, sc = r["labels"][0], float(r["scores"][0])
    pol = {"positive":+1.0,"negative":-1.0,"neutral":0.0}[lab]
    return float(np.clip(pol*sc, -1.0, 1.0))

try:
    term_df
except NameError:
    term_df = None

## Span(지속 길이) — 키워드/카테고리별 하프라이프 추정 + Term Impact Map

업데이트 루프(오프라인 배치): 이벤트 발생 후 관련 자산들의 실현 수익률 경로를 적합 →
단어별 polarity_mu/span_mu를 EWMA(alpha) 로 갱신 → term_impact_map.parquet 저장.

In [27]:
def span_from_terms(title: str):
    if term_df is None or term_df.empty:
        return None
    toks = re.findall(r"[A-Za-z가-힣0-9]+", title)
    vals = term_df[term_df["term"].isin(toks)]
    if len(vals)==0: return None
    return int(np.clip(vals["span_mu"].mean(), 1, CFG["span_max"]))

def default_span(cat: str, title: str) -> int:
    tl = title.lower()
    if cat=="exo":
        if any(k in tl for k in ["war","conflict","분쟁","전쟁"]): return 4
        if any(k in tl for k in ["fed","연준","금리"]): return 3
        if any(k in tl for k in ["oil","유가","gdp","성장률"]): return 2
        return 1
    else:
        if any(k in title for k in ["증자","수사","리콜","소송"]): return 2
        return 1

for f in row_feats:
    f["polarity"] = polarity_score(f["title"])
    sp = span_from_terms(f["title"])
    f["span"] = int(sp if sp else default_span(f["cat"], f["title"]))


- 검증 체크포인트: polarity 분포가 한쪽 치우치지 않음, span ≥1.
- 튜닝 포인트: 감성 템플릿을 문장형 유지, term map 보강 시 span_max 조절.

In [28]:
# 9.V — Polarity/Span sanity
pols = np.array([f["polarity"] for f in row_feats])
spans = np.array([f["span"] for f in row_feats])
print(f"[9.V] polarity min/mean/max = {pols.min():.3f}/{pols.mean():.3f}/{pols.max():.3f}")
print(f"span unique: {sorted(set(spans.tolist()))}")
assert (pols>=-1).all() and (pols<=1).all(), "polarity 범위 오류"
assert (spans>=1).all() and (spans<=CFG["span_max"]).all(), "span 범위 오류"


[9.V] polarity min/mean/max = -0.999/0.008/0.442
span unique: [1, 2, 3, 4]


# 일별→주간 TopN (LLM 보조 + MMR 재랭킹)

In [ ]:
from sentence_transformers import util

for f in row_feats:
    f["score_raw"] = 0.6*f["weight"] + 0.4*abs(f["polarity"])

def mmr_rows(cands, topk, lambda_, per_cluster_cap=1):
    if not cands or topk<=0:
        return []
    E = embed_texts([to_english(x["title"]) for x in cands])
    S = util.cos_sim(E,E).cpu().numpy()
    picked, cand_idx = [], list(range(len(cands)))
    scores = np.array([x["score_raw"] for x in cands])
    pick_count = {}
    while cand_idx and len(picked)<topk:
        valid = [ci for ci in cand_idx if pick_count.get(cands[ci]["cluster_id"],0) < per_cluster_cap]
        if not valid: break
        if not picked:
            chosen = valid[int(scores[valid].argmax())]
        else:
            rel = scores[valid]
            div = []
            for ci in valid:
                maxsim = max(S[ci, pj] for pj in picked) if picked else 0.0
                # 같은 클러스터면 벌점(유사도로 흡수)
                if cands[ci]["cluster_id"] in [cands[pj]["cluster_id"] for pj in picked]:
                    maxsim = max(maxsim, 0.9)
                div.append(maxsim)
            mmr = (1-lambda_)*rel - lambda_*np.array(div)
            chosen = valid[int(mmr.argmax())]
        picked.append(chosen)
        cand_idx.remove(chosen)
        cid = cands[chosen]["cluster_id"]
        pick_count[cid] = pick_count.get(cid,0)+1
    return [cands[i] for i in picked]

rows_exo  = [f for f in row_feats if f["cat"]=="exo"]
rows_endo = [f for f in row_feats if f["cat"]=="endo"]

top_exo  = mmr_rows(rows_exo,  CFG["weekly_topn_per_cat"]["exo"],  CFG["diversity_lambda"], CFG["per_cluster_cap"])
top_endo = mmr_rows(rows_endo, CFG["weekly_topn_per_cat"]["endo"], CFG["diversity_lambda"], CFG["per_cluster_cap"])

print("TopN exo/endo:", len(top_exo), len(top_endo))


TopN exo/endo: 1 1


- 검증 체크포인트: quota 충족(부족하면 파라미터 완화), 다양성 지표↑, cap 준수.
- 튜닝 포인트: 후보 부족→ per_cluster_cap=2, tau↓, dedup_threshold↓. 중복감→ diversity_lambda↑.

In [ ]:
# 10.V — TopN sanity
from sentence_transformers import util
def diversity(items):
    if len(items)<2: return 0.0
    E = embed_texts([to_english(x["title"]) for x in items])
    S = util.cos_sim(E,E).cpu().numpy()
    n = S.shape[0]
    return float((S.sum()-np.trace(S))/(n*(n-1)))

print(f"[10.V] top_exo={len(top_exo)} / quota={CFG['weekly_topn_per_cat']['exo']}, diversity={diversity(top_exo):.3f}")
print(f"[10.V] top_endo={len(top_endo)} / quota={CFG['weekly_topn_per_cat']['endo']}, diversity={diversity(top_endo):.3f}")

# per_cluster_cap 준수
def cap_ok(tops):
    from collections import Counter
    cnt = Counter([x["cluster_id"] for x in tops])
    return all(v <= CFG["per_cluster_cap"] for v in cnt.values())
assert cap_ok(top_exo) and cap_ok(top_endo), "per_cluster_cap 위반"

# (date,label) 중복 방지(TopN 내부)
def uniq_ok(items):
    seen=set()
    for x in items:
        k=f"{x['date']}|{x['title']}"
        if k in seen: return False
        seen.add(k)
    return True
assert uniq_ok(top_exo) and uniq_ok(top_endo), "TopN 중복 존재"


[10.V] top_exo=1 / quota=5, diversity=0.000
[10.V] top_endo=1 / quota=5, diversity=0.000


# CSV 생성(필수 5컬럼 + 보조컬럼 append)

In [ ]:
def to_rowdict(f, is_top=False, top_rank=None):
    base = dict(
        date=f["date"],
        label=f["title"],
        weight=float(np.clip(f["weight"],0.0,1.0)),
        polarity=float(np.clip(f["polarity"],-1.0,1.0)),
        span=int(max(0, f["span"])),
    )
    if CFG["output_append_extras"]:
        base.update(dict(
            cat=f["cat"], score_raw=f["score_raw"],
            cluster_id=f["cluster_id"], clust_size=f["clust_size"],
            src=f["source"], src_w=f["src_w"], centrality=f["centrality"],
            is_top=int(is_top), top_rank=(int(top_rank) if top_rank is not None else None),
        ))
    return base

def build_rows(all_rows, tops):
    topset = {id(x): i for i,x in enumerate(tops, start=1)}
    out = []
    for f in all_rows:
        is_top = id(f) in topset
        out.append(to_rowdict(f, is_top, topset.get(id(f))))
    return out

if CFG["core_contains"]=="top_only":
    ex_core = [to_rowdict(f, True, i+1) for i,f in enumerate(top_exo)]
    in_core = [to_rowdict(f, True, i+1) for i,f in enumerate(top_endo)]
else:
    ex_core = build_rows(rows_exo,  top_exo)
    in_core = build_rows(rows_endo, top_endo)

pd.DataFrame(ex_core).to_csv(CFG["out_ex"], index=False)
pd.DataFrame(in_core).to_csv(CFG["out_in"], index=False)
print("Wrote:", CFG["out_ex"], CFG["out_in"])


Wrote: events_ex.csv events_in.csv


# 유효성 검증(필수)

In [ ]:
def validate_core_csv(path: str) -> bool:
    df = pd.read_csv(path)
    # 필수 5컬럼 존재 확인(보조컬럼이 있어도 통과)
    must = ["date","label","weight","polarity","span"]
    for c in must:
        assert c in df.columns, f"{path}: 필수 컬럼 누락 {c}"
    # 타입/범위
    for s in df["date"]:
        _ = isoparse(str(s))
    assert (df["label"].astype(str).str.strip()!="").all()
    assert (df["weight"].between(0,1)).all()
    assert (df["polarity"].between(-1,1)).all()
    assert (df["span"].astype(int)>=0).all()
    # 권장: (date,label) 유일
    if len(df):
        assert df.assign(k=df["date"].astype(str)+"|"+df["label"]).drop_duplicates("k").shape[0]==len(df), "중복 (date,label)"
    return True

print("EXO valid:", validate_core_csv(CFG["out_ex"]))
print("ENDO valid:", validate_core_csv(CFG["out_in"]))


EXO valid: True
ENDO valid: True
